# GTÜ source adapter

Sections 1–6 run the steps in `gtue.adapter` (`1_scrapers/gtue/src/adapter.py`) against the saved sample response `1_scrapers/gtue/tests/fixtures/response_muenchen_50km_2026-09-28.json`. Section 7 fetches the finder with the adaptive grid (`core.grid`). All logic lives in the modules; this notebook only calls them. Every path comes from `core.paths`.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

import pandas as pd

from core import enrich, paths
from core.geo import ensure_geonames, load_plz_centroids
from gtue import adapter as gtue

ROOT = paths.project_root()
SAMPLE_PATH = paths.fixtures_dir("gtue") / "response_muenchen_50km_2026-09-28.json"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
ROOT

## 2. Load sample

In [ ]:
data = gtue.load_response(SAMPLE_PATH)

# The sample's fetch time is not recorded; the file's modification time is the best proxy.
scraped_at = datetime.fromtimestamp(SAMPLE_PATH.stat().st_mtime).isoformat(timespec="seconds")

print("Paging:", data["Paging"])
print("Query location:", data["SearchData"]["Location"])
print("scraped_at:", scraped_at)

## 3. Parse

In [ ]:
df = gtue.parse_response(data, scraped_at=scraped_at)
print(df.shape)
df.head()

### 3b. Enrich (GeoNames PLZ centroids)

Fills `state` from the PLZ, adds `plz_distance_km`, and flags rows that are far from their PLZ or have an unknown PLZ. Downloads `DE.zip` into `4_reference/geonames/` on first use.

In [ ]:
centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
print(f"{len(centroids)} PLZ centroids")

df = gtue.enrich_german(df, centroids)  # PLZ checks for German rows only
df[["source_id", "zip", "city", "state", "plz_distance_km", "needs_review", "needs_review_reason"]]

## 4. Validate

In [ ]:
report = gtue.validate(df, data)
print(gtue.format_report(report))

## 5. Inspect

In [ ]:
df[["name", "city", "herkunft", "inspects_on_site", "site_type"]]

In [ ]:
review = df[df["needs_review"]]
print(f"{len(review)} rows need review")
review[["source_id", "name", "city", "site_type_evidence"]]

### 5b. Scope tiers (rule of 2026-10-07)

`site_type` records what the finder's three signals say (herkunft, code `GTUP`, text "vor Ort"); **`scope_tier` decides `in_scope`**, following "if we do not have certain cause for exclusion, we include them":

| `scope_tier` | Meaning | `in_scope` |
|---|---|---|
| `pruefstelle` | E + code + text: own on-site inspection | yes |
| `unknown_conflict` | the three signals disagree | yes (`needs_review`) |
| `office_hu_listed` | G office (no on-site code) that lists *Hauptuntersuchung Pkw* | yes |
| `office_no_hu` | G office without any HU service | **no**: `scope_reason` = "no HU offered (finder service list)" |

Basis: hand checks of 5 + 5 G offices (5 of 5 without HU listed do none; 5 of 5 with HU listed do HU). Rows abroad stay out whatever the tier. `same_site_ids` lists the other German rows at the same site (same street + PLZ or within 150 m): GTÜ lists one site twice, as an E and a G row. Nothing is merged or dropped here; merging is the aggregator's job (`core.dedupe`).

## 6. Export

Writes the sample file's rows to a `recon` run folder of its own in `1_scrapers/gtue/test_output/` (`YYYY-MM-DD_HHMM_recon`). A rerun in the same minute gets the next free suffix (`_2`, `_3`, ...); nothing is ever overwritten.

In [ ]:
run_folder, written = gtue.write_recon_run(df, SAMPLE_PATH)
for key, (path, rows) in written.items():
    print(f"{key:<19}{rows:>5} rows  {path.relative_to(ROOT)}")

## 7. Fetch

Adaptive grid over Germany, queried at PLZ centroids (see `core/grid.py`). Each run prints its summary and writes one run folder `YYYY-MM-DD_HHMM_<mode>` with `raw.jsonl`, `query_log.csv`, `run_summary.json`/`.md`, the exports and, if squares stayed open, `open_squares.csv`: full runs in `1_scrapers/gtue/output/`, sample and simulated runs in `1_scrapers/gtue/test_output/`.

**Reference query.** Every run starts with one wide query: Zip 99986, City Niederdorla, 500 km (the server caps the radius at 500 km; the German border is at most 454 km from the centre, so the circle covers Germany). Its actual centre must be within 5 km of 51.15844 N, 10.44595 E, otherwise the run stops. A full run repeats it at the end; the sample run only queries it at the start. All values are constants at the top of the fetch section in `gtue/adapter.py` (`REFERENCE_*`).

**Run status (rule C4):** **PASS** needs both (a) every grid square complete (`Count == MaxCount` and the circle covers the square from the server's actual centre; 0 coverage warnings, 0 squares not queried, 0 errored queries) and (b) N_circle == the reference MaxCount, where N_circle = collected unique IDs within 500 km of the reference's actual centre (if the start and end MaxCount differ, anywhere in that range). Outcomes: match without rows abroad inside the circle = `PASS (grid complete + count match)`, the MaxCount is the German total; match with rows where `Country != "D"` inside the circle = PASS with a note, German total = rows with `Country == "D"`; N_circle below the MaxCount = FAIL (gap); N_circle above = FAIL, investigate. Rows with `Country != "D"` are never in scope. The drop check vs the last full run (alert if German locations drop more than 10 %) is reported next to the status and never changes it.

### 7a. Simulated run

No network: a fake API with 1,200 locations spread like PLZ centroids. It must find all 1,200 and report PASS.

Stress test (optional): `gtue.run(mode="simulated", centroids=centroids, sim_clusters=grid.BIG_CITIES)` adds 60 locations within 10 km of each of five big cities. It still finds all 1,200 but reports FAIL on purpose: more than 20 locations fall inside the smallest circles (coverage warnings).

In [ ]:
sim = gtue.run(mode="simulated", centroids=centroids)

### 7b. Sample run (real API, 5 requests)

Self-contained: runs right after a kernel restart without the cells above (own imports, loads `.env`, own config). Needs `GTUE_API_KEY` in `.env`; the cell only prints whether it is set, never the key.

Request 1 is the reference query (99986 Niederdorla, 500 km; it stops the run if its centre is more than 5 km off), requests 2–3 are the cap probe (MaxCount for München 80686 at 455 and 460 km; equal counts mean the server cap has dropped below what covers Germany, so the early stop would be switched off), requests 4–5 are 2 grid squares. A sample always reports **FAIL** (most squares are not queried, and it has no end query); check the numbers, not the status. The summary shows the reference MaxCount and centre, the cap probe, N_circle (collected IDs within 500 km of that centre), rows abroad inside and outside the circle, the German total, the grid figures, and every query whose server centre is more than 5 km from the PLZ centroid (PLZ, offset, square). The query log shows `center_offset_km` and a timestamp per request. Everything goes to one run folder in `1_scrapers/gtue/test_output/` (raw responses in its `raw.jsonl`).

In [ ]:
# Self-contained: runs right after a kernel restart, without the cells above
import os

from dotenv import load_dotenv

from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from gtue import adapter as gtue

ROOT = paths.project_root()
load_dotenv(ROOT / ".env", override=True)

# Config
MODE = "sample"  # reference query + cap probe (2) + 2 grid squares = 5 requests
RUN_DATE = None  # None = today
print(f"{gtue.API_KEY_ENV} set: {bool(os.environ.get(gtue.API_KEY_ENV))}")  # never print the key itself

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
sample = gtue.run(mode=MODE, root=ROOT, run_date=RUN_DATE, centroids=centroids)
sample.query_log

### 7c. Full run

Use of GTÜ's finder key is approved by management (2026-09-28). The full run needs `confirm_full_run=True`. The committed default is `False`: with it, the cell stops with an error on purpose before any request is sent.

Self-contained like 7b. Request budget: the safety limit is 2,000 live requests (the reference queries, the cap probe and re-queries count; answers from the same-day cache do not). Offline replay of the 2026-09-29 run: with the re-query from the actual centre the whole grid needs roughly 1,750–1,950 requests (about 3.5 s each, i.e. 1.7–2 h); a re-run on the same date reuses that day's responses (from the `raw.jsonl` of the earlier runs of that day in `output/`; the summary says which run folders) and needs only a few hundred new ones. The run status follows rule C4 (section 7). The drop check compares German locations with the most recent full run in `output/` whose `run_summary.json` says PASS; on the first full run it reports "no baseline". A full run that is not a PASS writes its files with `grid-incomplete` (or `count-mismatch`) in the names and lists the open squares in `open_squares.csv` in its run folder.

`EARLY_STOP` (default `False`, turn it on after the first full run): stop the grid once N_circle reaches the reference MaxCount, re-checked once against the reference, and report `PASS (early stop, count match)` even with squares still open (they stay listed). It first runs a cap probe (München 80686 at 455 and 460 km, 2 extra requests); equal counts switch it off for that run and the summary says so.

`USE_CACHE` (default `True`): `False` ignores the same-day cache, so every query (repeats included) is sent live. Every run has a run folder of its own, so no earlier raw file is touched either way. The summary then shows `cache: off`. A fresh run needs roughly 1,900 requests.

`MAX_REQUESTS` (default `2000`): safety limit on live requests. If it is hit, the run stops with a clear message, the status is FAIL (never PASS) and the raw responses so far are kept in the run's `raw.jsonl` (see 7d).

In [ ]:
# Self-contained: runs right after a kernel restart, without the cells above
import os

from dotenv import load_dotenv

from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from gtue import adapter as gtue

ROOT = paths.project_root()
load_dotenv(ROOT / ".env", override=True)

# Config
RUN_DATE = None  # None = today
EARLY_STOP = False  # rule E; turn on after the first full run
USE_CACHE = True  # False = empty cache: every query is sent live
MAX_REQUESTS = 2000  # safety limit on live requests; if hit, the run stops and is never a PASS
print(f"{gtue.API_KEY_ENV} set: {bool(os.environ.get(gtue.API_KEY_ENV))}")  # never print the key itself

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
# Set confirm_full_run=True to start the full run. Keep False in the committed notebook.
full = gtue.run(
    mode="full", confirm_full_run=False, root=ROOT, run_date=RUN_DATE, centroids=centroids, early_stop=EARLY_STOP,
    use_cache=USE_CACHE, max_requests=MAX_REQUESTS,
)

### 7d. Rebuild from raw (no network)

Reads the `raw.jsonl` of a run folder in `1_scrapers/gtue/output/`, dedupes by ID, classifies, enriches and exports into a new run folder of its own (`YYYY-MM-DD_HHMM_rebuild`, next to its source run); its `run_summary.json`/`.md` name the source run. It gets no copy of `raw.jsonl`. Sends no request and never overwrites (a rerun in the same minute gets `_2`, ...).

Two uses:
- **Re-classify a run that passed** (a scope rule changed, e.g. 2026-10-07). If the source run's `run_summary.json` says PASS and the rebuild finds exactly its unique and German location counts, the rebuild inherits the PASS: outcome `PASS (rebuilt from <run>, re-classified, no network; ...)`, no tag in the file names. The aggregator accepts such a rebuild of a passing run.
- **Rescue a run that stopped before the grid was complete** (e.g. the request limit). Then the files carry `_grid-incomplete` and the summary is headed **GRID INCOMPLETE**: not the result of a verified complete crawl.

If the first raw response is a reference query (centre within 5 km of Niederdorla), it also prints N_circle against that MaxCount; runs from before the reference query began with a national check from elsewhere and get no count check.

In [ ]:
# Self-contained: runs right after a kernel restart, without the cells above. No network, no API key needed.
from core import paths
from core.geo import ensure_geonames, load_plz_centroids
from gtue import adapter as gtue

# Config
SOURCE_RUN = "2026-09-29_1019_full"  # a run folder name under 1_scrapers/gtue/output/

centroids = load_plz_centroids(ensure_geonames(paths.geonames_dir()))
rebuilt = gtue.rebuild_from_raw(SOURCE_RUN, centroids)